# Lab 24: The Stroop Tangle

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 3, *The Attention Filter*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-24.ipynb)

**What you will do:** time yourself on the three Stroop lists, turn your times into milliseconds of interference per word, compare them with published values, explore a model of the conflict, and test whether an AI vision model shows its own Stroop-like confusion.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 24 you read three lists aloud as fast as you could (Figure 3.3 in the book):
**List A** — colour words in black ink (read the words); **List B** — rows of XXXX in coloured
ink (name the colours); **List C** — colour words in *mismatching* ink (name the ink colour and
ignore the word). List C is slower because the automatic word-reading response has to be
suppressed.

## Record your results

Type your times (in seconds) into the cell below and run it. Figure 3.3 has 12 items per
list; change `items_per_list` if you used your own lists.

In [ ]:
items_per_list = 12
my_times_s = {"A: read words": 5.1, "B: name colours (XXXX)": 7.0, "C: name colours (conflict)": 9.6}  # example data: replace with yours
my_errors = {"A: read words": 0, "B: name colours (XXXX)": 0, "C: name colours (conflict)": 2}      # example data

results = pd.DataFrame({"time (s)": my_times_s, "errors": my_errors})
results["ms per item"] = (results["time (s)"] * 1000 / items_per_list).round(0)
display(results)
results["ms per item"].plot.barh(color=["grey", "steelblue", "firebrick"], figsize=(6, 2.4))
plt.xlabel("milliseconds per item"); plt.gca().invert_yaxis(); plt.show()

## Compare

The Stroop **interference** is the extra time per item in List C compared with List B (both
ask you to name colours; only C has conflicting words). Lab 24 reports that laboratory
studies typically find interference of about 100–200 ms per response.

In [ ]:
interference = results.loc["C: name colours (conflict)", "ms per item"] - results.loc["B: name colours (XXXX)", "ms per item"]
print(f"Your interference: {interference:.0f} ms per item")
if interference < 100:
    print("Below the typical 100-200 ms range: did you slow down on List B, or practise List C first?")
elif interference <= 200:
    print("Inside the typical 100-200 ms range reported in Lab 24.")
else:
    print("Above the typical range. Note that timing a whole list aloud (with its hesitations and"
          " self-corrections) is not the same measurement as the single-trial reaction times"
          " behind the 100-200 ms figure, so treat the comparison as rough.")

## Simulation

The Stroop effect slows you down because two predictions (word meaning and ink colour) compete for the same response. This simulation models two processing pathways—a fast, automatic word-reading route and a slower, controlled colour-naming route—and shows how conflict between them produces the reaction-time delay you experienced.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_trials = 500

# Word-reading pathway: fast, automatic (mean 450 ms)
word_rt = np.random.normal(450, 40, n_trials)

# Colour-naming pathway: slower, controlled (mean 550 ms)
colour_rt = np.random.normal(550, 50, n_trials)

# Congruent trials: word and ink colour agree, so the fast automatic
# route can drive the response with little extra cost
congruent = word_rt + np.random.normal(0, 15, n_trials)

# Incongruent trials: the automatic word-reading response must be
# suppressed before the controlled colour-naming response can proceed
conflict_penalty = np.abs(np.random.normal(60, 20, n_trials))
incongruent = colour_rt + conflict_penalty

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(300, 900, 60)
ax.hist(congruent, bins, alpha=0.5, color="green",
    label=f"Congruent (mean {np.mean(congruent):.0f} ms)")
ax.hist(incongruent, bins, alpha=0.5, color="red",
    label=f"Incongruent (mean {np.mean(incongruent):.0f} ms)")
ax.axvline(np.mean(congruent), color="green", linestyle="--",
    linewidth=2)
ax.axvline(np.mean(incongruent), color="red", linestyle="--",
    linewidth=2)
ax.set_xlabel("Reaction time (ms)")
ax.set_ylabel("Number of trials")
ax.set_title("Stroop Conflict: Two Predictions, One Response")
ax.legend()
plt.tight_layout()
plt.show()

effect = np.mean(incongruent) - np.mean(congruent)
print(f"Stroop effect: {effect:.0f} ms")

**What to notice:** The red distribution (incongruent) is shifted rightward by roughly 160 ms, but that whole gap is not the cost of conflict. About 100 ms of it is simply the built-in speed difference between the two pathways (colour naming is modelled as slower than word reading even with no conflict at all); only the remaining ~60 ms, from `conflict_penalty`, is the modelled cost of suppressing the automatic (but wrong) word-reading response before the controlled colour-naming response can proceed. Real Stroop experiments—which compare colour-naming with and without a conflicting word, holding the pathway constant—typically find interference in the range of 100–200 ms (MacLeod, 1991), so treat this simulated conflict cost as illustrative rather than a direct read-off of that figure.

## Change the parameters

**conflict_mean** and **conflict_sd** control the size and variability of the penalty
added on incongruent trials -- the cost of suppressing the automatic word-reading
response. **word_mean** and **colour_mean** control how fast each pathway is on its own.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider

def explore(conflict_mean=60, conflict_sd=20, word_mean=450, colour_mean=550):
    np.random.seed(42)
    n_trials = 500
    word_rt = np.random.normal(word_mean, 40, n_trials)
    colour_rt = np.random.normal(colour_mean, 50, n_trials)
    congruent = word_rt + np.random.normal(0, 15, n_trials)
    conflict_penalty = np.abs(np.random.normal(conflict_mean, conflict_sd, n_trials))
    incongruent = colour_rt + conflict_penalty

    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.linspace(300, 950, 60)
    ax.hist(congruent, bins, alpha=0.5, color="green", label=f"Congruent (mean {np.mean(congruent):.0f} ms)")
    ax.hist(incongruent, bins, alpha=0.5, color="red", label=f"Incongruent (mean {np.mean(incongruent):.0f} ms)")
    ax.axvline(np.mean(congruent), color="green", linestyle="--", linewidth=2)
    ax.axvline(np.mean(incongruent), color="red", linestyle="--", linewidth=2)
    ax.set_xlabel("Reaction time (ms)")
    ax.set_ylabel("Number of trials")
    ax.set_title("Stroop Conflict: Two Predictions, One Response")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Modelled Stroop effect: {np.mean(incongruent) - np.mean(congruent):.0f} ms")

interact(explore,
         conflict_mean=IntSlider(value=60, min=0, max=150, step=10),
         conflict_sd=IntSlider(value=20, min=5, max=60, step=5),
         word_mean=IntSlider(value=450, min=350, max=600, step=10),
         colour_mean=IntSlider(value=550, min=400, max=700, step=10));

## The AI side

Some AI models show a Stroop-like conflict too. CLIP, a model that matches images with text,
can be misled by written words in a picture: shown an apple with a label reading "iPod", it
may call it an iPod. Researchers call this a **typographic attack** (Goh et al., 2021,
*Distill*, "Multimodal neurons in artificial neural networks"). Below, we show CLIP colour
words printed in matching and mismatching ink and ask it which ink colour it sees. Watch
whether the written word pulls its answer away from the real ink colour, as it pulls yours.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
INKS = {"red": (200, 16, 46), "blue": (0, 114, 178), "green": (0, 158, 115), "yellow": (212, 177, 6)}

def card(text, ink):
    img = Image.new("RGB", (224, 224), "white")
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 44)
    except OSError:
        font = ImageFont.load_default()
    draw = ImageDraw.Draw(img)
    w = draw.textlength(text, font=font)
    draw.text(((224 - w) / 2, 88), text, fill=INKS[ink], font=font)
    return img

prompts = [f"text written in {c} ink" for c in INKS]
rows = []
for word in INKS:
    for ink in INKS:
        inputs = processor(text=prompts, images=card(word.upper(), ink), return_tensors="pt", padding=True)
        with torch.no_grad():
            probs = model(**inputs).logits_per_image.softmax(dim=1)[0].numpy()
        choice = list(INKS)[probs.argmax()]
        rows.append({"word": word.upper(), "ink": ink, "congruent": word == ink,
                     "CLIP says": choice, "correct": choice == ink,
                     "pulled to word": (choice == word) and (word != ink)})
ai = pd.DataFrame(rows)
display(ai)
print("CLIP accuracy, congruent cards:  ", f"{ai[ai.congruent].correct.mean():.0%}")
print("CLIP accuracy, incongruent cards:", f"{ai[~ai.congruent].correct.mean():.0%}")
print("Incongruent cards where CLIP named the WORD instead of the ink:",
      f"{ai[~ai.congruent]['pulled to word'].mean():.0%}")

Compare the two accuracies. If CLIP is worse on incongruent cards, and its errors name the
printed word, it is showing a machine analogue of Stroop interference. The analogy has
limits: CLIP has no reaction time and nothing to suppress; it simply combines evidence from
the letters and the colour in one step. (Results can differ slightly between model versions.)

## Pool the class data

One person's times are noisy. Pool the class: each person adds one row (an anonymous ID, not a name, plus List B and List C
times in seconds) to a shared spreadsheet, downloads it as CSV and loads it here. The
example rows below are made-up placeholders so the cell runs.

In [ ]:
import io
csv_text = """id,B_seconds,C_seconds
P01,7.2,9.9
P02,6.4,8.1
P03,8.0,10.7
P04,6.9,8.8
P05,7.5,9.4
"""  # example data: replace with your class CSV, e.g. pd.read_csv("stroop_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
df["interference_ms"] = (df.C_seconds - df.B_seconds) * 1000 / items_per_list
boot = [df.interference_ms.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean interference: {df.interference_ms.mean():.0f} ms per item "
      f"(95% bootstrap interval {lo:.0f}-{hi:.0f} ms, n = {len(df)})")
df.interference_ms.plot.hist(bins=8, color="firebrick", alpha=0.7, figsize=(6, 3))
plt.axvspan(100, 200, color="grey", alpha=0.2, label="typical range (Lab 24)")
plt.xlabel("interference (ms per item)"); plt.legend(); plt.show()

## Questions to think about

1. Set conflict_mean to 0. What happens to the modelled Stroop effect, and does this match the claim that the automatic word-reading response "must be actively suppressed" before colour naming can proceed?
2. The lab reports that real Stroop interference typically falls between 100 and 200 ms. Adjust conflict_mean until the printed effect lands near the middle of that range. What value did you need?
3. Make word_mean and colour_mean equal, so neither pathway is faster on its own. Does the congruent-incongruent gap disappear, shrink, or stay about the same? What does this tell you about whether the Stroop delay comes from a difference in pathway speed or from the cost of suppression?
4. Increase conflict_sd a lot while keeping conflict_mean fixed. What happens to how much the two histograms overlap, even though their means barely move?

## Challenge

Run a version of the Stroop test on yourself (the printed lists, the smartphone app, or a
quick homemade List A/List C from Step 1 of the lab), timing your own congruent and
incongruent reading aloud. Compare your measured interference with the model above.

In [ ]:
# Example data: replace with your own stopwatch timings, in milliseconds, for reading
# several congruent and incongruent lists aloud.
my_congruent_ms = [820, 795, 830, 810, 805]      # example data
my_incongruent_ms = [980, 1010, 960, 1005, 990]  # example data

my_effect = np.mean(my_incongruent_ms) - np.mean(my_congruent_ms)
print(f"Your measured Stroop effect: {my_effect:.0f} ms")
print("Compare this with the value the simulation prints above, and with the 100-200 ms")
print("range reported in the wider literature (MacLeod, 1991).")
print("The two lists above are example data -- replace them with your own timings.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-24.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")